# Building the Bolivia 343 municipal GDP panel

Reproduce the estimates used by `geometrics.data.load_bolivia343()`: fixed
343-unit municipality/TIOC geometry, 2012–2022, allocated from the existing
0.25° PWT-anchored GRID case study. These are **modeled allocations**, not
official municipal GDP. Keep this notebook's outputs local: the boundary map
is attributed to SDSN Bolivia and is not relicensed by geometrics. Do not
publish copies of this notebook containing output geometry without appropriate
permission.

Walkthrough: verify sources → run one shared construction program → inspect
the allocation and sensitivity ledgers → check accounting independently →
compare modeled departmental shares with official INE accounts (optional).

## 1. Open the notebook in Colab and select the source

**Before release:** In a fresh Colab runtime, run the next cell and upload a
ZIP of your geometrics checkout, including `pyproject.toml`, `uv.lock`,
`src/` and `tools/`. Your ZIP must include the
version of the code that came with this notebook. The ZIP stays in the
runtime; no GitHub credentials, unpublished branch, or publishing step is
needed. A ZIP of *only the notebook or a patch* is not sufficient. For
example, with a prepared checkout named `geometrics`, run this from its
parent directory:

```sh
zip -r geometrics-colab.zip geometrics/pyproject.toml geometrics/uv.lock   geometrics/README.md geometrics/LICENSE geometrics/src geometrics/tools   geometrics/datasets/BOL-005popAdj-PWTscaled
```

**After release:** Instead of uploading, explicitly set `GEOMETRICS_REF` to an
available public release tag or full commit SHA. For a non-Colab worktree,
set `LOCAL_REPO`. By default outputs and the loader cache are temporary.
Pinned public GRID files are downloaded into the temporary runtime if not in the ZIP.
Fresh Colab needs network access to install dependencies and retrieve verified
official reference files unless exact local copies are supplied. No outputs
are published by this notebook.

In [ ]:
import hashlib
import shutil
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

LOCAL_REPO = None  # Example: "/home/me/geometrics"
SOURCE_ZIP = None  # Otherwise prompted to upload in Colab.
GEOMETRICS_REF = None  # Optional released public tag or full commit SHA.
OUTPUT_DIR = None  # Optional private output directory.
MAP343_LOCAL = None  # Optional exact map343.geojson to avoid downloading it.
DEPENDENCY_PYTHON = None  # Optional Python from an already installed geometrics environment.

run_dir = Path(tempfile.mkdtemp(prefix="geometrics-bolivia343-")).resolve()

def repo_candidates(directory):
    return sorted(
        p.parent.parent for p in directory.rglob("tools/build_bolivia_adm3.py")
        if (p.parent.parent / "pyproject.toml").is_file()
    )

if LOCAL_REPO:
    repo = Path(LOCAL_REPO).expanduser().resolve()
elif GEOMETRICS_REF:
    repo = run_dir / "geometrics"
    subprocess.run(["git", "init", "-q", str(repo)], check=True)
    subprocess.run(
        ["git", "-C", str(repo), "fetch", "--depth", "1",
         "https://github.com/quarcs-lab/geometrics.git", GEOMETRICS_REF], check=True,
    )
    subprocess.run(["git", "-C", str(repo), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
else:
    if SOURCE_ZIP is None:
        try:
            from google.colab import files
        except ImportError as exc:
            raise RuntimeError("Set LOCAL_REPO or SOURCE_ZIP when running outside Colab.") from exc
        uploads = files.upload()
        names = [name for name in uploads if name.lower().endswith(".zip")]
        if len(names) != 1:
            raise ValueError("Upload exactly one geometrics source ZIP.")
        SOURCE_ZIP = run_dir / Path(names[0]).name
        SOURCE_ZIP.write_bytes(uploads[names[0]])
    archive = Path(SOURCE_ZIP).expanduser().resolve()
    destination = run_dir / "source"
    destination.mkdir()
    with zipfile.ZipFile(archive) as zf:
        if any(not (destination / item).resolve().is_relative_to(destination) for item in zf.namelist()):
            raise ValueError("Source ZIP contains an unsafe path.")
        zf.extractall(destination)
    candidates = repo_candidates(destination)
    if len(candidates) != 1:
        raise ValueError("Expected one full geometrics source directory in the ZIP.")
    repo = candidates[0]
    print("Source ZIP SHA-256:", hashlib.sha256(archive.read_bytes()).hexdigest())

build = repo / "tools" / "build_bolivia_adm3.py"
if not build.is_file():
    raise FileNotFoundError("Select a geometrics release or ZIP containing tools/build_bolivia_adm3.py.")
if (repo / ".git").exists():
    print("geometrics commit:", subprocess.check_output(
        ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
    ).strip())
    if subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain"], text=True).strip():
        print("Working-tree changes are present; preserve the working tree for an exact replay.")
out = Path(OUTPUT_DIR).expanduser().resolve() if OUTPUT_DIR else run_dir / "ADM3"
print("Working source:", repo)

## 2. Input provenance and allocation rule

- **GRID:** Rossi-Hansberg & Zhang, [Local GDP estimates around the world](https://doi.org/10.1016/j.jue.2026.103871)
  ([BFI data](https://bfidatastudio.org/gdp)); geometrics' 0.25° `0_05` GRID
  release at commit `c23a78de67db2a72c3243c43887927675b685c7c`. In this Bolivia
  input, flagged low-density cells have zero estimated GDP, not necessarily zero population.
  GDP and population already sum to [PWT 11.0](https://www.rug.nl/ggdc/productivity/pwt/)
  `rgdpo` and `pop`, respectively. The units are million 2021 US dollars at
  chained PPPs and million persons.
- **Boundaries:** SDSN Bolivia, [Límites Municipales Bolivia 2025](https://sdsnbolivia.org/datos-espaciales/),
  obtained from the [GeoDevelopment Observatory's pinned map343](https://github.com/quarcs-lab/gdo4bolivia/blob/9a2e2b536b743700a69ca8b682ba57feb9bf7e26/geo/boundaries/map343.geojson).
  This is reference geography, not a legal boundary decision. The Observatory
  documents its permission from SDSN Bolivia; this notebook does not presume
  further redistribution rights. New municipal/TIOC footprints are held constant
  across historical observations, not treated as historical administrations.

The [source data definitions](https://bfidatastudio.org/wp-content/uploads/2026/05/Data-Definition.pdf)
identify longitude and latitude as **southwest corners**, not centroids. The
legacy geometry was shifted half a cell south and west. The builder reconstructs
full 0.25° envelopes from these corners; it does not recreate the publisher's
exact clipped border/water polygons or alter original GRID values.

For the main assignment, measure the intersection of a full grid square with
each municipal polygon in an equal-area projection. For nonzero total
covered area, define the weight as that intersection divided by the sum of
intersections for the cell. Multiply both GDP and population by the same
weight. Aggregated GDP per capita is municipal total GDP divided by municipal
total population, not an average of contributing cell ratios. This area rule
assumes a uniform within-cell estimate; it does not use a population surface.
The full-square and centroid routes are assumptions to compare, not error
limits. The original cell-level random-forest uncertainty statistics are
not municipal confidence intervals.

In [ ]:
import hashlib
import os

import urllib.request

grid_ref = "c23a78de67db2a72c3243c43887927675b685c7c"
dataset_rel = Path("datasets/BOL-005popAdj-PWTscaled")
bundled = repo / dataset_rel
dataset = run_dir / "verified-inputs"
(dataset / "GRID").mkdir(parents=True, exist_ok=True)
source_hashes = {
    "GRID/bolivia_grid_cells.csv":
        "3af614177f84322211ead2366822f1baa6ad074773ae774de1515ce26e82f2e2",
    "GRID/bolivia_grid_cells.gpkg":
        "26df0f506929cc041969311fcbd69f369d24a1f99d890a05a564cf78e8971040",
    "GRID/bolivia_grid_cells_data_def.csv":
        "7c0096e975c90a27c204db8224f5abecc05299ec6f412adf020d5f2cf32ce998",
}
for relative, expected in source_hashes.items():
    path = dataset / relative
    supplied = bundled / relative
    if supplied.is_file():
        # Copy rather than moving or changing user's files.
        shutil.copyfile(supplied, path)
    else:
        url = f"https://raw.githubusercontent.com/quarcs-lab/geometrics/{grid_ref}/{dataset_rel}/{relative}"
        with urllib.request.urlopen(url, timeout=60) as response:
            path.write_bytes(response.read())
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f"Unexpected input hash for {path.name}: {actual}")
    print(f"{path.name}: {actual}")

# Keep geospatial dependencies isolated from the running Colab kernel.
if DEPENDENCY_PYTHON:
    uv_run = [str(Path(DEPENDENCY_PYTHON).expanduser().absolute())]
else:
    uv = shutil.which("uv")
    if uv is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        uv = shutil.which("uv") or str(Path(sys.executable).with_name("uv"))
    assert Path(uv).is_file(), "uv is not on PATH. Install uv, then rerun this cell."
    uv_run = [uv, "run", "--locked", "--no-dev", "python"]
build_cmd = [*uv_run, str(build), "--datasets-root", str(dataset), "--output", str(out)]
if MAP343_LOCAL is not None:
    build_cmd += ["--map343", str(Path(MAP343_LOCAL).expanduser().resolve())]
source_env = {**os.environ, "PYTHONPATH": str(repo / "src")}
subprocess.run(build_cmd, cwd=repo, check=True, env=source_env)
print(f"Audit outputs: {out}")
print("Harmonized panel API: geometrics.data.load_bolivia_harmonized(level)")

## 3. Read the coverage, allocation sensitivity and topology diagnostics

Intersections are measured in equal-area EPSG:6933; overlaps of 1 m² or
less are ignored. The full-cell denominator retains only the estimated mass proportionate to
covered area, with the unallocated portion identified in the ledger. Main
covered-share normalization redistributes this mass to the intersecting
municipalities; inspect low coverage carefully before using a border town.
The centroid comparison assigns a cell to the municipality covering its
center; cells whose centers fall outside the map are left unassigned. No method resolves activity within the original grid squares.
A consistent national total is not evidence of municipal accuracy.

In [ ]:
import json

import numpy as np
import pandas as pd
from IPython.display import display

panel = pd.read_csv(out / "bolivia_adm3.csv")
ledger = pd.read_csv(out / "allocation_ledger.csv")
weights = pd.read_csv(out / "allocation_cell_weights.csv")
sensitivity = pd.read_csv(out / "sensitivity.csv")
diagnostics = json.loads((out / "validation.json").read_text())
assert diagnostics["grid_geometry"] == "full quarter-degree envelope from documented southwest corner"
assert diagnostics["upstream_clipped_border_polygons_used"] is False
print(json.dumps(diagnostics, indent=2, ensure_ascii=False))
audit_table = ledger.assign(
    unassigned_gdp_pct=100 * ledger.unassigned_gdp_pwt / ledger.source_gdp_pwt,
    unassigned_pop_pct=100 * ledger.unassigned_pop_pwt / ledger.source_pop_pwt,
)
display(audit_table.loc[audit_table.year.eq(2022), ["method", "unassigned_gdp_pct", "unassigned_pop_pct"]])
display(weights.sort_values(["covered_area_fraction", "cell"]).head(8))
pc22 = sensitivity.loc[sensitivity.year.eq(2022)].pivot(index="geo_id", columns="method", values="gdppc")
rows = []
for method in ("full_footprint", "centroid"):
    common = pc22[["covered_area", method]].dropna()
    rows.append({
        "method": method,
        "matching finite GDPpc estimates": len(common),
        "rank correlation": common.corr(method="spearman").iloc[0, 1],
        "GDPpc differs by >10%": (((common[method] - common.covered_area).abs() / common.covered_area.where(common.covered_area.gt(0))).gt(.1)).sum(),
        "median absolute GDPpc difference": (common[method] - common.covered_area).abs().median(),
    })
display(pd.DataFrame(rows))

## 4. Independently re-sum the outputs

The source panel is read separately here, not taken from the builder's
validation loop. Matching GDP and population totals is an accounting check
against the inherited national anchor, not an external observation of
municipal GDP. Recompute the reported per capita value from additive
quantities. All 343 footprints are in the panel; inspect the ledger before
interpreting a zero or missing value as a local economic fact.

In [ ]:
municipality_id = "geo_id"
assert panel[municipality_id].nunique() == 343
assert not panel.duplicated([municipality_id, "year"]).any()
assert panel["year"].sort_values().unique().tolist() == list(range(2012, 2023))
assert panel.groupby("year").size().eq(343).all()
totals = ["gdp_pwt", "pop_pwt"]
supported = panel["coverage_status"].ne("no_support")
assert panel.loc[supported, totals].notna().all().all()
assert panel.loc[supported, totals].ge(0).all().all()
assert panel.loc[~supported, totals].isna().all().all()

key_parts = ["cell_id", "subcell_id", "subcell_id_0_25"]
grid = pd.read_csv(dataset / "GRID" / "bolivia_grid_cells.csv", usecols=["year", *key_parts, *totals])
grid["cell"] = grid[key_parts].astype(str).agg(".".join, axis=1)
linked = weights.loc[weights.geo_id.notna()]
np.testing.assert_allclose(linked.groupby("cell").weight_covered_area.sum(), 1, atol=1e-12)
assert set(weights.cell) == set(grid.cell)
uncovered = grid.loc[~grid.cell.isin(linked.cell)]
grid_totals = grid.groupby("year")[totals].sum()
unassigned = uncovered.groupby("year")[totals].sum().reindex(grid_totals.index, fill_value=0)
assigned = panel.groupby("year")[totals].sum().reindex(grid_totals.index)
balance = grid_totals - assigned - unassigned
np.testing.assert_allclose(assigned + unassigned, grid_totals, rtol=1e-10, atol=1e-10)
display(pd.concat({"original GRID": grid_totals, "assigned ADM3": assigned, "unassigned": unassigned, "balance error": balance}, axis=1))

geometry_check = """
import sys
import geopandas as gpd
import pandas as pd
gdf = gpd.read_file(sys.argv[1])
df = pd.read_csv(sys.argv[2])
assert gdf.crs.to_epsg() == 4326
assert len(gdf) == 343 and gdf.geo_id.is_unique and set(gdf.geo_id) == set(df.geo_id)
assert gdf.is_valid.all() and not gdf.is_empty.any()
print("All 343 geometry keys match the panel; valid nonempty EPSG:4326 polygons.")
"""
subprocess.run(
    [*uv_run, "-c", geometry_check, str(out / "bolivia_adm3_boundaries.gpkg"), str(out / "bolivia_adm3.csv")],
    cwd=repo, check=True,
)

positive_pop = panel["pop_pwt"].gt(0)
positive_gdppc = panel["gdppc"].gt(0)
np.testing.assert_allclose(
    panel.loc[positive_pop, "gdppc"],
    panel.loc[positive_pop, "gdp_pwt"] / panel.loc[positive_pop, "pop_pwt"],
    rtol=1e-8,
    atol=1e-9,
)
np.testing.assert_allclose(
    panel.loc[positive_gdppc, "ln_gdppc"], np.log(panel.loc[positive_gdppc, "gdppc"]),
    rtol=1e-8, atol=1e-9,
)
assert panel.loc[~positive_pop, "gdppc"].isna().all()
assert panel.loc[~positive_gdppc, "ln_gdppc"].isna().all()

## 5. Same-year external check: the 2012 population census

The independent release check uses [INE table 3.09.02.02](https://nimbus.ine.gob.bo/index.php/s/EgN8PPYD8Y4Rdig)
and the [pinned geo4bolivia/GDO SDG population data](https://github.com/quarcs-lab/gdo4bolivia/blob/9a2e2b536b743700a69ca8b682ba57feb9bf7e26/docs/datasets/sdg.md)
on the same map343 geography. The 343 municipal counts match the 2012 INE
release; verified parent crosswalks give 113 provinces and nine departments.
The four post-2012 units refer to their modern footprints. The 2024 census is
outside this panel, not 2022 truth.

| Scale | Rank correlation, legacy → corrected | Median absolute % error, legacy → corrected |
|---|---:|---:|
| Municipality / TIOC | 0.649 → 0.822 | 54.05 → 38.24 |
| Province | 0.787 → 0.944 | 46.22 → 19.54 |
| Department | 0.983 → 1.000 | 9.29 → 4.94 |

The publisher's coordinate definitions, not a fit to census, identify the
correction. The corrected municipality counts are still over 25% from the
census for 227 of 343 units. This is a population check, not GDP validation or
proof that the LandScan-based inputs share no census information. For the
checks from the build itself, see the
[ADM3 verification notes](https://github.com/quarcs-lab/geometrics/blob/main/datasets/BOL-005popAdj-PWTscaled/ADM3/VERIFICATION.md).

In [ ]:
RUN_CENSUS_CHECK = False  # Set True to retrieve the pinned GDO and INE source tables.
CENSUS_WORKBOOK_LOCAL = None  # Optional exact local INE workbook.
GDO_LOCAL = None  # Optional exact local GDO data tree.
if RUN_CENSUS_CHECK:
    census_out = out / "ine_census_2012"
    census_python = uv_run if DEPENDENCY_PYTHON else [*uv_run[:-1], "--with", "openpyxl", "python"]
    cmd = [
        *census_python, str(repo / "tools/validate_bolivia_census.py"),
        "--adm3", str(out / "bolivia_adm3.csv"), "--out", str(census_out),
    ]
    if CENSUS_WORKBOOK_LOCAL is not None:
        cmd += ["--ine-workbook", str(Path(CENSUS_WORKBOOK_LOCAL).expanduser().resolve())]
    if GDO_LOCAL is not None:
        cmd += ["--gdo-local", str(Path(GDO_LOCAL).expanduser().resolve())]
    subprocess.run(cmd, cwd=repo, check=True)
    census = json.loads((census_out / "summary.json").read_text())
    display(pd.DataFrame(census["same_year_level_metrics"]).T.round(3))

## 6. Population denominators: annual INE, exact 2012 census, and PWT control

The area allocation above still constructs **GDP**, unchanged. The headline
`gdppc` divides that modeled GDP by **INE annual population** in people
(`pop_ine`). GDP in millions is multiplied by a million before division. Do not call this series an annual headcount or official municipal
GDP per capita. [INE yearbook table 2.01.12](https://anuario.ine.gob.bo/2023/documentos/201/20112.xlsx) is used consistently for
2012–2022. [INE Revisión 2020, §7(d)](https://siip.produccion.gob.bo/noticias/files/2021-934cb-1estimaINE.pdf#page=44) uses the 2012 census to adjust its municipal shares. Treat the 2012 comparison as census agreement, not an independent holdout, even for the 333 one-to-one matches. [INE Revisión 2025](https://www.ine.gob.bo/index.php/censos-y-proyecciones-de-poblacion-sociales/) starts in 2024 at the municipality/TIOC level.
The numerator is anchored to PWT output-side real GDP at chained PPPs, not national-accounts growth.

- **333 directly represented units:** one-to-one mapped INE yearbook
  projections (`ine_yearbook_projection`).
- **10 units in four historical groups:** the yearbook does not publish these
  modern areas separately. Preserve each annual group total and split it using
  within-group shares computed from the exact 2012 and 2024 censuses, linearly
  interpolated between census years (`census_share_interpolated`). The
  largest-remainder method rounds people to integers while preserving annual
  group totals exactly. The 2024
  census only helps locate the split and is outside the GDP panel. The exact
  2012 census and 2012 yearbook projection are distinct.
- **2012 census comparison:** divide the same GDP by exact INE 2012 headcounts
  (`pop_census_2012`) as a same-year sensitivity only. The census denominator
  and `gdppc_census_2012` must be missing outside 2012.
- **PWT control:** `gdppc_pwt` uses the original area-allocated `pop_pwt` in
  millions. Keep it for comparisons to the national PWT scaling, not as the
  default municipal denominator.

This cell reproduces annual totals, checks the same numerator in all ratios,
and displays the published yearbook counts alongside the exact census and
PWT population. The bundled harmonized population is constructed from
official INE sources. To rebuild it from original workbooks, run
`python tools/build_bolivia_population.py --output pop.csv --metadata pop.json`
with pandas and openpyxl installed. It verifies downloaded source checksums;
use `--help` for offline workbook overrides.
A closer population denominator does not validate the GDP numerator. Large
GDP-per-capita changes can expose municipalities where the modeled GDP and
the independently measured residential population are hard to reconcile.

In [ ]:
# Stage the immutable GRID source and the verified map when supplied so the loader
# uses its normal checksum policy without redundant downloads.
import os
cache = run_dir / "source-cache"
(cache / "GRID").mkdir(parents=True, exist_ok=True)
for basename in ("bolivia_grid_cells.csv", "bolivia_grid_cells.gpkg", "bolivia_grid_cells_data_def.csv"):
    shutil.copyfile(dataset / "GRID" / basename, cache / "GRID" / basename)
if MAP343_LOCAL is not None:
    (cache / "BOL343").mkdir(exist_ok=True)
    shutil.copyfile(Path(MAP343_LOCAL).expanduser().resolve(), cache / "BOL343" / "map343.geojson")
verify_population = """
import sys
from pathlib import Path
from geometrics.data import load_bolivia343
from geometrics.data._bolivia_population import load_bolivia343_population
_, panel, dictionary = load_bolivia343()
published = load_bolivia343_population()
p = Path(sys.argv[1])
panel.to_csv(p / 'population_denominator_panel.csv', index=False)
dictionary.to_csv(p / 'population_denominator_dictionary.csv', index=False)
published.to_csv(p / 'ine_population_reference.csv', index=False)
"""
subprocess.run([*uv_run, "-c", verify_population, str(out)], cwd=repo, check=True,
               env={**os.environ, "GEOMETRICS_DATA_DIR": str(cache),
                    "PYTHONPATH": str(repo / "src")})
headline = pd.read_csv(out / "population_denominator_panel.csv")
ine_source = pd.read_csv(out / "ine_population_reference.csv")
assert len(headline) == len(ine_source) == 3773
key = ["geo_id", "year"]
method_map = {
    "ine_yearbook_projection": 333,
    "census_share_interpolated": 10,
}
assert ine_source.groupby("year").pop_ine_method.value_counts().unstack().eq(pd.Series(method_map)).all().all()
split = ine_source[ine_source.pop_ine_method.eq("census_share_interpolated")]
assert split["pop_ine_group"].nunique() == 4
assert sorted(split.groupby("pop_ine_group").geo_id.nunique()) == [2, 2, 3, 3]
population_meta = json.loads((repo / "src/geometrics/data/bolivia343_population_metadata.json").read_text())
observed_groups = split.pivot_table(index="pop_ine_group", columns="year",
                                    values="pop_ine_harmonized", aggfunc="sum").astype("Int64")
for group in population_meta["split_groups"]:
    group_id = "+".join(group["historical_geo_ids"])
    np.testing.assert_array_equal(observed_groups.loc[group_id],
                                 group["annual_projection_total_people"])
display(observed_groups)
# All denominators use the untouched same-dollar GDP allocation from Section 4.
same = headline.merge(panel[key + ["gdp_pwt", "pop_pwt", "gdppc"]],
                      on=key, suffixes=("", "_area_only"), validate="one_to_one")
for column in ["gdp_pwt", "pop_pwt"]:
    np.testing.assert_allclose(same[column], same[column + "_area_only"], equal_nan=True)
np.testing.assert_allclose(same.gdppc_pwt, same.gdppc_area_only, equal_nan=True)
check = headline.merge(ine_source, on=key, suffixes=("", "_source"), validate="one_to_one")
assert check.pop_ine_method.eq(check.pop_ine_method_source).all()
assert check.pop_ine_group.eq(check.pop_ine_group_source).all()
np.testing.assert_allclose(check.pop_ine, check.pop_ine_harmonized)
np.testing.assert_allclose(check.gdppc, 1e6 * check.gdp_pwt / check.pop_ine, equal_nan=True)
y12 = check.year.eq(2012)
assert check.loc[~y12, ["pop_census_2012", "gdppc_census_2012"]].isna().all().all()
np.testing.assert_allclose(check.loc[y12, "pop_census_2012"],
                           check.loc[y12, "pop_ine_census2012"])
np.testing.assert_allclose(check.loc[y12, "gdppc_census_2012"],
                           1e6 * check.loc[y12, "gdp_pwt"] / check.loc[y12, "pop_census_2012"])
expected_yearbook = np.array([
    10356978, 10521247, 10685994, 10851103, 11016438, 11181861,
    11347241, 11512468, 11677406, 11841955, 12006031,
])
national = check.groupby("year")[["pop_ine", "pop_pwt", "pop_census_2012"]].sum(min_count=1)
national["pop_pwt"] *= 1e6
assert national.loc[2012, "pop_census_2012"] == 10_059_856
np.testing.assert_allclose(national.pop_ine, expected_yearbook, rtol=0, atol=1e-6)
display(national.round(0).astype("Int64").rename(columns={
    "pop_ine": "INE annual harmonized (people)",
    "pop_pwt": "PWT area allocation (people)",
    "pop_census_2012": "INE exact census (people)",
}))
# Census agreement for the denominator; not an independent holdout or GDP validation.
municipal_2012 = check.loc[y12].copy()
for column in ["pop_ine", "pop_pwt"]:
    error = (municipal_2012[column] * (1e6 if column == "pop_pwt" else 1) / municipal_2012.pop_census_2012 - 1).abs()
    print(f"2012 median absolute population difference versus census, {column}: {100 * error.median():.2f}%")
print("Largest INE-denominator estimates below are diagnostic flags, not validated municipal GDP per capita.")
display(municipal_2012[["geo_id", "name", "gdppc", "gdppc_pwt", "gdppc_census_2012"]]
        .sort_values("gdppc", ascending=False).head(10))

## 7. Optional external comparison: departmental GDP shares

[INE's official table D.10.1.2](https://www.ine.gob.bo/referencia2017/pib_departamental.html)
publishes departmental shares of nominal GDP; 2022 is marked preliminary in
the pinned workbook. Compare it with the shares of modeled, PPP real GDP for
2022 as a **broad spatial plausibility check**. Never compare the monetary
levels or interpret the differences as municipal measurement errors. The
department mapping below uses the Observatory's IDs, not similarly ordered
INE codes. INE was not used to calibrate this ADM3 allocation.

The [official INE census tables](https://www.ine.gob.bo/index.php/estadisticas-sociales/poblacion-censo/)
are a separate candidate population check. Align the footprints and years;
2024 census data do not validate modeled 2022 population. GDP per capita is
sensitive to error in both component allocations.

In [ ]:
import unicodedata
import urllib.request

RUN_EXTERNAL_CHECK = False  # Set True and rerun this cell to try the external check.
INE_WORKBOOK_PATH = None  # Optional local copy of the exact workbook below.
INE_URL = "https://www.ine.gob.bo/referencia2017/CUADROS/D.10.1.2.xlsx"
INE_SHA256 = "6ba69e9a27138809d1758441510001f31415fefa5eeb5820352b95a3a71f0907"
DEPARTMENT = {
    "BO-01": "Chuquisaca", "BO-02": "Cochabamba", "BO-03": "Beni",
    "BO-04": "La Paz", "BO-05": "Oruro", "BO-06": "Pando",
    "BO-07": "Potosí", "BO-08": "Santa Cruz", "BO-09": "Tarija",
}

def normalize_name(name):
    return "".join(
        c for c in unicodedata.normalize("NFKD", name.casefold())
        if not unicodedata.combining(c)
    ).strip()

if RUN_EXTERNAL_CHECK:
    import io
    from urllib.error import URLError

    if INE_WORKBOOK_PATH:
        workbook = Path(INE_WORKBOOK_PATH).expanduser().read_bytes()
    else:
        try:
            with urllib.request.urlopen(INE_URL, timeout=30) as response:
                workbook = response.read()
        except (URLError, TimeoutError) as error:
            workbook = None
            print(f"Optional INE download unavailable: {error}")
    if workbook is not None:
        if hashlib.sha256(workbook).hexdigest() != INE_SHA256:
            raise ValueError("INE revised the workbook or returned another file. Do not silently change the benchmark.")
        official = pd.read_excel(io.BytesIO(workbook), sheet_name="D.10.1.2", header=None, engine="openpyxl")
        departments_2022 = pd.DataFrame({"name": official.iloc[13:22, 1], "INE nominal GDP share (%)": official.iloc[13:22, 7]})
        departments_2022["dept_key"] = departments_2022["name"].map(normalize_name)
        departments_2022["INE nominal GDP share (%)"] = departments_2022["INE nominal GDP share (%)"].astype(float)
        np.testing.assert_allclose(departments_2022["INE nominal GDP share (%)"].sum(), 100)

        latest = panel.loc[panel["year"].eq(2022), [municipality_id, "gdp_pwt"]].copy()
        latest["dept_key"] = latest[municipality_id].str.slice(0, 5).map(DEPARTMENT).map(normalize_name)
        assert latest["dept_key"].notna().all()
        shares = 100 * latest.groupby("dept_key")["gdp_pwt"].sum() / latest["gdp_pwt"].sum()
        comparison = departments_2022.merge(
            shares.rename("Model PPP GDP share (%)"), on="dept_key", validate="one_to_one"
        ).drop(columns="dept_key")
        assert len(comparison) == 9
        comparison["Model minus INE (pp)"] = comparison["Model PPP GDP share (%)"] - comparison["INE nominal GDP share (%)"]
        display(comparison.round(3))
        print("Spearman rank correlation:", comparison.iloc[:, 1:3].corr(method="spearman").iloc[0, 1].round(3))
        print("Mean absolute share difference (pp):", comparison["Model minus INE (pp)"].abs().mean().round(3))

        import plotly.express as px

        share_columns = ["Model PPP GDP share (%)", "INE nominal GDP share (%)"]
        figure = px.bar(
            comparison.sort_values(share_columns[1], ascending=False), x="name", y=share_columns,
            barmode="group", title="Bolivia, 2022: modeled and official departmental GDP shares",
            labels={"name": "Department", "value": "% of national GDP", "variable": "Basis"},
        )
        display(figure)